# Music Annotation Project V8.2a
## Policy Sensitivity, Threshold Sweep and Decision Overlay Audit

**Runtime:** CPU is sufficient; GPU is not required.

V8.2a does **not retrain, refit or recalibrate** the V8.2 model. It finds one completed V8.2 development run, verifies and copies its durable models, calibrators, inputs, audit files and result tables into the V8.2a run, then performs a development-only policy sensitivity analysis on the saved OOF probabilities.

### Outputs added by V8.2a
- a threshold sweep using the unchanged saved V8.2 probabilities;
- probability distributions and quantiles;
- a clearly labelled tweakable policy block;
- a chosen **primary exploratory policy** used to produce V8.2a whole pages;
- annotated-page overlays using **green = PRINT**, **red = ANNOTATION**, **amber = UNCERTAIN**;
- source V8.2 models/calibrators/files of record copied into the V8.2a directory with SHA-256 inventory;
- V7.8 fallback and outside-mask integrity assertions retained.

### Important interpretation
The registered V8.2 rule remains the historical comparison point. V8.2a is a new, post-outcome **development sensitivity audit**, not a frozen challenger and not independent evidence. Any promising policy found here would need a separately declared version and fresh validation before promotion.

### Binary-rule interaction
For a binary model, `p_annotation = 1 - p_print`, so the margin condition is:

`p_print - p_annotation >= margin` → `p_print >= (1 + margin) / 2`.

Therefore lowering `PRINT_THRESHOLD` alone has no effect if `PRINT_MARGIN` remains at `0.990`; that margin still imposes an effective threshold of `0.995`. V8.2a records the effective threshold explicitly and sweeps coordinated threshold/margin pairs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib


## 1. Configuration and clearly identifiable tuning controls


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, sys, platform, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
V82_ROOT = ROOT / 'observed_print_classifier_v8_2'
OUTPUT_ROOT = ROOT / 'observed_print_classifier_v8_2a'

# Optional: point to one exact completed V8.2 run. Leave as None to use the latest complete run.
SOURCE_V82_RUN_OVERRIDE = None

# ============================================================
# V8.2a TWEAKABLE POLICY CONTROLS
# ============================================================

# Primary exploratory policy used to create V8.2a whole pages.
PRIMARY_PRINT_THRESHOLD = 0.980
PRIMARY_PRINT_MARGIN = 0.960       # binary-equivalent to p_print >= 0.980
PRIMARY_ANNOTATION_THRESHOLD = 0.995
PRIMARY_ENABLE_SINGLETON_FILTER = True
PRIMARY_MIN_PRINT_RUN = 2

# Sweep coordinated probability thresholds. Margin is generated as 2*t - 1,
# so threshold and margin represent the same effective binary cut-off.
SWEEP_PRINT_THRESHOLDS = [
    0.995, 0.990, 0.985, 0.980, 0.975,
    0.950, 0.925, 0.900, 0.850, 0.800,
]
SWEEP_SINGLETON_FILTER_OPTIONS = [True, False]
SWEEP_ANNOTATION_THRESHOLD = 0.995
BOOTSTRAP_RESAMPLES = 10000
SEED = 42

# Overlay controls.
OVERLAY_ALPHA = 0.48
OVERLAY_REVIEW_RADIUS = 1          # 1 => 3x3 visibility aid; exact overlay also saved
GENERATE_PRINT_ONLY_OVERLAY = True

# Record-copy controls. 'complete_core' copies inputs/models/calibrators/audit/tables
# plus root-level identity/summary files. Whole-page source images are copied per page.
SOURCE_RECORD_MODE = 'complete_core'

# ============================================================

if not (0.5 <= PRIMARY_PRINT_THRESHOLD <= 1.0):
    raise ValueError('PRIMARY_PRINT_THRESHOLD must be between 0.5 and 1.0.')
if not (0.0 <= PRIMARY_PRINT_MARGIN <= 1.0):
    raise ValueError('PRIMARY_PRINT_MARGIN must be between 0.0 and 1.0.')
if not (0.5 <= PRIMARY_ANNOTATION_THRESHOLD <= 1.0):
    raise ValueError('PRIMARY_ANNOTATION_THRESHOLD must be between 0.5 and 1.0.')
if PRIMARY_MIN_PRINT_RUN < 1:
    raise ValueError('PRIMARY_MIN_PRINT_RUN must be at least 1.')

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v8_2a_policy_sensitivity'
OUT = OUTPUT_ROOT / RUN
RECORD = OUT / 'source_v82_record'
TABLES = OUT / 'tables'
WHOLE = OUT / 'whole_pages'
VISUAL = OUT / 'visual_audit'
AUDIT = OUT / 'audit'
for p in [OUT, RECORD, TABLES, WHOLE, VISUAL, AUDIT]:
    p.mkdir(parents=True, exist_ok=False)

print('V8.2a output:', OUT)


## 2. Find one completed V8.2 source run and validate the required records


In [ ]:
REQUIRED_SOURCE_FILES = [
    'experiment_identity.json',
    'development_summary.json',
    'executive_summary.txt',
    'tables/all_eligible_oof_predictions.parquet',
    'audit/model_inventory.csv',
    'audit/calibrator_inventory.csv',
    'inputs/fold_map.csv',
    'inputs/channel_schema.json',
]

def find_complete_runs(root):
    hits=[]
    if root.exists():
        for p in root.iterdir():
            if p.is_dir() and all((p/r).exists() for r in REQUIRED_SOURCE_FILES):
                hits.append(p)
    return sorted(hits, key=lambda p:p.stat().st_mtime)

if SOURCE_V82_RUN_OVERRIDE is not None:
    SOURCE = Path(SOURCE_V82_RUN_OVERRIDE)
    missing=[r for r in REQUIRED_SOURCE_FILES if not (SOURCE/r).exists()]
    if missing:
        raise RuntimeError(f'SOURCE_V82_RUN_OVERRIDE is incomplete: {missing}')
else:
    runs=find_complete_runs(V82_ROOT)
    if not runs:
        raise RuntimeError(f'No complete V8.2 source run found below {V82_ROOT}')
    SOURCE=runs[-1]

source_identity=json.loads((SOURCE/'experiment_identity.json').read_text())
if source_identity.get('version') != '8.2-development-whole-page':
    raise RuntimeError(f'Unexpected source version: {source_identity.get("version")}')
if source_identity.get('independent_data_used') is not False:
    raise RuntimeError('V8.2a must operate on the development-only V8.2 OOF run.')

pred_path=SOURCE/'tables'/'all_eligible_oof_predictions.parquet'
full=pd.read_parquet(pred_path)
required_columns={'page','source_group','y','x','truth','fold','raw_print','p_print'}
missing=required_columns-set(full.columns)
if missing:
    raise RuntimeError(f'Source predictions lack required columns: {sorted(missing)}')
for c in ['y','x','truth','fold']:
    full[c]=pd.to_numeric(full[c],errors='raise').astype(int)
for c in ['raw_print','p_print']:
    full[c]=pd.to_numeric(full[c],errors='raise').astype(np.float64)
if not np.isfinite(full[['raw_print','p_print']].to_numpy()).all():
    raise RuntimeError('Source V8.2 predictions contain non-finite scores.')
if full.duplicated(['page','y','x']).any():
    raise RuntimeError('Source V8.2 predictions contain duplicate page/y/x rows.')

print('Source V8.2 run:', SOURCE)
print('Eligible rows:', len(full))
print('Pages:', full.page.nunique())
print('Source groups:', full.source_group.nunique())


## 3. Copy the V8.2 models, calibrators and files of record into V8.2a


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):
            h.update(b)
    return h.hexdigest()

def copy_item(src,dst):
    if src.is_dir():
        shutil.copytree(src,dst)
    else:
        dst.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(src,dst)

# Complete core record: no dependency on a later search for the V8.2 model/calibrator/config.
for d in ['inputs','models','calibrators','audit','tables']:
    src=SOURCE/d
    if not src.exists():
        raise RuntimeError(f'Missing source V8.2 record directory: {src}')
    copy_item(src,RECORD/d)

for name in ['experiment_identity.json','development_summary.json','executive_summary.txt','software_environment.json']:
    src=SOURCE/name
    if src.exists():
        copy_item(src,RECORD/name)

# Validate copied model and calibrator hashes against the source experiment identity.
for item in source_identity.get('models',[]):
    src_name=Path(item['checkpoint']).name
    copied=RECORD/'models'/src_name
    if not copied.exists() or sha256_file(copied)!=item['checkpoint_sha256']:
        raise RuntimeError(f'Copied model hash mismatch: {src_name}')
for item in source_identity.get('calibrators',[]):
    src_name=Path(item['joblib']).name
    copied=RECORD/'calibrators'/src_name
    if not copied.exists() or sha256_file(copied)!=item['joblib_sha256']:
        raise RuntimeError(f'Copied calibrator hash mismatch: {src_name}')

inventory=[]
for p in sorted(RECORD.rglob('*')):
    if p.is_file():
        inventory.append({
            'relative_path':str(p.relative_to(RECORD)),
            'bytes':p.stat().st_size,
            'sha256':sha256_file(p),
        })
pd.DataFrame(inventory).to_csv(AUDIT/'copied_source_v82_artifact_inventory.csv',index=False)

source_record={
    'source_run':str(SOURCE),
    'source_version':source_identity.get('version'),
    'source_created_utc':source_identity.get('created_utc'),
    'source_manifest_sha256':source_identity.get('manifest_snapshot_sha256'),
    'source_fold_map_sha256':source_identity.get('fold_map_sha256'),
    'source_channel_schema_sha256':source_identity.get('channel_schema_sha256'),
    'copied_files':len(inventory),
    'copied_bytes':int(sum(x['bytes'] for x in inventory)),
    'source_predictions_sha256':sha256_file(pred_path),
    'copied_predictions_sha256':sha256_file(RECORD/'tables'/'all_eligible_oof_predictions.parquet'),
    'model_hashes_verified':True,
    'calibrator_hashes_verified':True,
}
(OUT/'source_v82_record.json').write_text(json.dumps(source_record,indent=2))
print(json.dumps(source_record,indent=2))


## 4. Probability-distribution audit


In [ ]:
quantile_grid=[0,0.001,0.01,0.05,0.10,0.25,0.50,0.75,0.90,0.95,0.975,0.99,0.995,0.999,1.0]
quantile_rows=[]
for label,df in [('ALL',full),('ANNOTATION_TRUTH',full[full.truth==0]),('PRINT_TRUTH',full[full.truth==1]),('UNCERTAIN_TRUTH',full[full.truth==2])]:
    if len(df)==0: continue
    q=df.p_print.quantile(quantile_grid)
    for prob,val in q.items():
        quantile_rows.append({'population':label,'quantile':float(prob),'p_print':float(val),'rows':len(df)})
quantiles=pd.DataFrame(quantile_rows)
quantiles.to_csv(TABLES/'probability_quantiles.csv',index=False)

bins=np.r_[np.linspace(0,0.95,96),[0.96,0.97,0.975,0.98,0.985,0.99,0.995,0.9975,0.999,0.9995,0.9999,1.0000001]]
hist_rows=[]
for truth,name in [(0,'ANNOTATION'),(1,'PRINT'),(2,'UNCERTAIN')]:
    vals=full.loc[full.truth==truth,'p_print'].to_numpy()
    counts,edges=np.histogram(vals,bins=bins)
    for lo,hi,n in zip(edges[:-1],edges[1:],counts):
        hist_rows.append({'truth':name,'bin_left':float(lo),'bin_right':float(hi),'count':int(n)})
pd.DataFrame(hist_rows).to_csv(TABLES/'probability_histogram.csv',index=False)

plt.figure(figsize=(10,6))
for truth,name in [(0,'ANNOTATION'),(1,'PRINT'),(2,'UNCERTAIN')]:
    vals=full.loc[full.truth==truth,'p_print'].to_numpy()
    plt.hist(vals,bins=100,histtype='step',label=name,log=True)
plt.xlabel('Calibrated PRINT probability')
plt.ylabel('Pixel count (log scale)')
plt.title('V8.2 OOF calibrated probability distribution')
plt.legend(); plt.tight_layout(); plt.savefig(VISUAL/'probability_distribution.png',dpi=180); plt.close()
print(quantiles[quantiles['quantile'].isin([.9,.95,.975,.99,.995,.999,1.0])].to_string(index=False))


## 5. Policy helpers, page-global run filter and registered metrics


In [ ]:
def effective_print_threshold(print_threshold, print_margin):
    return max(float(print_threshold),(1.0+float(print_margin))/2.0)

def provisional_decision(prob,print_threshold,print_margin,annotation_threshold):
    p=np.asarray(prob,dtype=np.float64); q=1.0-p
    out=np.full(len(p),'UNCERTAIN',object)
    out[q>=annotation_threshold]='ANNOTATION'
    out[(p>=print_threshold)&((p-q)>=print_margin)]='PRINT'
    return out

def apply_run_filter(df, provisional, enabled=True, min_run=2):
    final=np.asarray(provisional,dtype=object).copy()
    demoted=0
    if not enabled or min_run<=1:
        return final,demoted
    for page,g in df.groupby('page',sort=False):
        idx=g.index.to_numpy()
        h=int(g.y.max())+1; w=int(g.x.max())+1
        pm=np.zeros((h,w),np.uint8)
        pidx=idx[provisional[idx]=='PRINT']
        if len(pidx)==0: continue
        pm[df.loc[pidx,'y'].to_numpy(),df.loc[pidx,'x'].to_numpy()]=1
        n,lab,stats,_=cv2.connectedComponentsWithStats(pm,8)
        for cid in range(1,n):
            if int(stats[cid,cv2.CC_STAT_AREA])<min_run:
                ys,xs=np.where(lab==cid)
                keys={(int(y),int(x)) for y,x in zip(ys,xs)}
                drop=[i for i in pidx if (int(df.at[i,'y']),int(df.at[i,'x'])) in keys]
                final[drop]='UNCERTAIN'; demoted+=len(drop)
    return final,demoted

def wilson_upper(errors,n,z=1.96):
    if n==0:return np.nan
    p=errors/n; d=1+z*z/n; c=p+z*z/(2*n); r=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)); return (c+r)/d

# Authoritative-mask component IDs are rebuilt from the exact V7.5 reference named by V8.2.
v75=Path(source_identity['v7_5_reference'])
status=pd.read_csv(v75/'tables'/'cache_status.csv')
cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
component_maps={}
mask_maps={}
for page in sorted(full.page.unique()):
    if page not in cache: raise RuntimeError(f'V7.5 cache mapping missing for {page}')
    with np.load(cache[page]) as z:
        if 'mask' not in z.files: raise RuntimeError(f'{page}: source cache lacks mask')
        mask=z['mask']>0
    _,labels,_,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
    mask_maps[page]=mask
    component_maps[page]=labels

def summarise_policy(df,decisions,policy_id,print_threshold,print_margin,annotation_threshold,singleton_filter,min_run,demoted,bootstrap_resamples=BOOTSTRAP_RESAMPLES):
    work=df[['page','source_group','y','x','truth','p_print']].copy(); work['decision']=decisions
    selected=work.decision=='PRINT'; scored=work.truth.isin([0,1]); tp=int((selected&(work.truth==1)).sum()); fp=int((selected&(work.truth==0)).sum()); tu=int((selected&(work.truth==2)).sum())
    component_rows=[]
    for page,g in work[selected].groupby('page'):
        labels=component_maps[page]
        cids=labels[g.y.to_numpy(),g.x.to_numpy()]
        g=g.copy(); g['component_id']=cids
        for cid,q in g[g.component_id>0].groupby('component_id'):
            ctp=int((q.truth==1).sum()); cfp=int((q.truth==0).sum()); ctu=int((q.truth==2).sum())
            catastrophic=bool(cfp>=50 or (cfp/max(ctp,1)>1.0 and cfp>=20))
            component_rows.append({'policy_id':policy_id,'page':page,'source_group':q.source_group.iloc[0],'component_id':int(cid),'selected':len(q),'true_print':ctp,'false_annotation':cfp,'truth_uncertain':ctu,'catastrophic':catastrophic})
    component_columns=['policy_id','page','source_group','component_id','selected','true_print','false_annotation','truth_uncertain','catastrophic']
    comps=pd.DataFrame(component_rows,columns=component_columns)
    cat=int(comps.catastrophic.sum()) if len(comps) else 0
    page_rows=[]
    for page,g in work.groupby('page'):
        s=g.decision=='PRINT'; pt=int((s&(g.truth==1)).sum()); pf=int((s&(g.truth==0)).sum()); page_cat=int(((comps.page==page)&comps.catastrophic).sum()) if len(comps) else 0
        page_rows.append({'policy_id':policy_id,'page':page,'source_group':g.source_group.iloc[0],'eligible':len(g),'selected':int(s.sum()),'true_print':pt,'false_annotation':pf,'truth_uncertain_print':int((s&(g.truth==2)).sum()),'utility':pt-5*pf-500*page_cat,'catastrophic_components':page_cat,'print_coverage':float(s.mean()),'uncertain_rate':float((g.decision=='UNCERTAIN').mean())})
    pages=pd.DataFrame(page_rows)
    rng=np.random.default_rng(SEED); unique=pages.page.to_numpy(); indexed=pages.set_index('page'); boot=[]
    for _ in range(bootstrap_resamples):
        q=indexed.loc[rng.choice(unique,size=len(unique),replace=True)]; bt=int(q.true_print.sum()); bf=int(q.false_annotation.sum()); bn=bt+bf; bc=int(q.catastrophic_components.sum()); boot.append((bt-5*bf-500*bc,bf/max(bt,1),bf/max(bn,1)))
    b=np.asarray(boot,float); selected_scored=int((selected&scored).sum())
    summary={
        'policy_id':policy_id,'print_threshold':print_threshold,'print_margin':print_margin,'effective_print_threshold':effective_print_threshold(print_threshold,print_margin),'annotation_threshold':annotation_threshold,'singleton_filter':bool(singleton_filter),'min_print_run':int(min_run),'singleton_demotions':int(demoted),'eligible_pixels':len(work),'selected_print_pixels':int(selected.sum()),'true_print':tp,'false_annotation':fp,'truth_uncertain_print':tu,'false_true_ratio':fp/max(tp,1),'false_preservation_rate':fp/max(selected_scored,1),'wilson95_upper_false_preservation':wilson_upper(fp,selected_scored),'utility':tp-5*fp-500*cat,'catastrophic_components':cat,'benefiting_pages':int((pages.true_print>=10).sum()),'benefiting_source_groups':int(pages.loc[pages.true_print>=10,'source_group'].nunique()),'print_coverage':float(selected.mean()),'uncertain_rate':float((work.decision=='UNCERTAIN').mean()),'bootstrap_utility_95_low':float(np.quantile(b[:,0],.025)),'bootstrap_utility_95_high':float(np.quantile(b[:,0],.975)),'bootstrap_false_true_95_high':float(np.quantile(b[:,1],.975)),'bootstrap_false_preservation_95_high':float(np.quantile(b[:,2],.975)),
    }
    return summary,pages,comps,work

print('Policy helpers ready.')


## 6. Threshold and singleton-filter sweep using unchanged V8.2 probabilities


In [ ]:
sweep_summaries=[]; sweep_pages=[]; sweep_components=[]
for threshold in SWEEP_PRINT_THRESHOLDS:
    margin=max(0.0,2.0*float(threshold)-1.0)
    for singleton_filter in SWEEP_SINGLETON_FILTER_OPTIONS:
        policy_id=f'p{threshold:.4f}_m{margin:.4f}_run{PRIMARY_MIN_PRINT_RUN if singleton_filter else 1}'
        provisional=provisional_decision(full.p_print.to_numpy(),threshold,margin,SWEEP_ANNOTATION_THRESHOLD)
        decisions,demoted=apply_run_filter(full,provisional,singleton_filter,PRIMARY_MIN_PRINT_RUN)
        summary,pages,comps,_=summarise_policy(full,decisions,policy_id,threshold,margin,SWEEP_ANNOTATION_THRESHOLD,singleton_filter,PRIMARY_MIN_PRINT_RUN,demoted)
        sweep_summaries.append(summary); sweep_pages.append(pages)
        if len(comps): sweep_components.append(comps)

sweep=pd.DataFrame(sweep_summaries).sort_values(['singleton_filter','effective_print_threshold'],ascending=[False,False])
sweep.to_csv(TABLES/'threshold_sweep_summary.csv',index=False)
pd.concat(sweep_pages,ignore_index=True).to_csv(TABLES/'threshold_sweep_by_page.csv',index=False)
if sweep_components:
    pd.concat(sweep_components,ignore_index=True).to_csv(TABLES/'threshold_sweep_components.csv',index=False)
else:
    pd.DataFrame().to_csv(TABLES/'threshold_sweep_components.csv',index=False)

plt.figure(figsize=(10,6))
for flag,label in [(True,'singleton filter on'),(False,'singleton filter off')]:
    q=sweep[sweep.singleton_filter==flag].sort_values('effective_print_threshold')
    plt.plot(q.effective_print_threshold,q.true_print,marker='o',label=f'True PRINT — {label}')
    plt.plot(q.effective_print_threshold,q.false_annotation,marker='x',label=f'False annotation — {label}')
plt.xlabel('Effective PRINT threshold'); plt.ylabel('Selected scored pixels'); plt.title('V8.2a threshold sensitivity'); plt.legend(); plt.tight_layout(); plt.savefig(VISUAL/'threshold_sweep_true_false.png',dpi=180); plt.close()

plt.figure(figsize=(10,6))
for flag,label in [(True,'singleton filter on'),(False,'singleton filter off')]:
    q=sweep[sweep.singleton_filter==flag].sort_values('effective_print_threshold')
    plt.plot(q.effective_print_threshold,q.utility,marker='o',label=label)
plt.axhline(0,color='black',linewidth=1); plt.xlabel('Effective PRINT threshold'); plt.ylabel('Net safety utility'); plt.title('V8.2a utility sensitivity'); plt.legend(); plt.tight_layout(); plt.savefig(VISUAL/'threshold_sweep_utility.png',dpi=180); plt.close()

print(sweep[['policy_id','effective_print_threshold','singleton_filter','selected_print_pixels','true_print','false_annotation','truth_uncertain_print','utility','catastrophic_components','benefiting_pages','benefiting_source_groups']].to_string(index=False))


## 7. Apply the primary exploratory policy


In [ ]:
PRIMARY_POLICY_ID=f'primary_p{PRIMARY_PRINT_THRESHOLD:.4f}_m{PRIMARY_PRINT_MARGIN:.4f}_run{PRIMARY_MIN_PRINT_RUN if PRIMARY_ENABLE_SINGLETON_FILTER else 1}'
primary_provisional=provisional_decision(full.p_print.to_numpy(),PRIMARY_PRINT_THRESHOLD,PRIMARY_PRINT_MARGIN,PRIMARY_ANNOTATION_THRESHOLD)
primary_decisions,primary_demoted=apply_run_filter(full,primary_provisional,PRIMARY_ENABLE_SINGLETON_FILTER,PRIMARY_MIN_PRINT_RUN)
primary_summary,primary_pages,primary_components,primary_rows=summarise_policy(full,primary_decisions,PRIMARY_POLICY_ID,PRIMARY_PRINT_THRESHOLD,PRIMARY_PRINT_MARGIN,PRIMARY_ANNOTATION_THRESHOLD,PRIMARY_ENABLE_SINGLETON_FILTER,PRIMARY_MIN_PRINT_RUN,primary_demoted)

primary_rows['raw_print']=full.raw_print.to_numpy(); primary_rows['fold']=full.fold.to_numpy(); primary_rows['decision_pre_run_rule']=primary_provisional
primary_rows.to_parquet(TABLES/'primary_policy_predictions.parquet',index=False)
primary_rows.to_csv(TABLES/'primary_policy_predictions.csv',index=False)
primary_pages.to_csv(TABLES/'primary_results_by_page.csv',index=False)
primary_components.to_csv(TABLES/'primary_selected_components.csv',index=False)
primary_components[primary_components.catastrophic==True].to_csv(TABLES/'primary_catastrophic_components.csv',index=False)
(OUT/'primary_policy_summary.json').write_text(json.dumps(primary_summary,indent=2))

policy_config={
    'policy_id':PRIMARY_POLICY_ID,'development_sensitivity_only':True,'source_v82_run':str(SOURCE),'print_threshold':PRIMARY_PRINT_THRESHOLD,'print_margin':PRIMARY_PRINT_MARGIN,'effective_print_threshold':effective_print_threshold(PRIMARY_PRINT_THRESHOLD,PRIMARY_PRINT_MARGIN),'annotation_threshold':PRIMARY_ANNOTATION_THRESHOLD,'enable_singleton_filter':PRIMARY_ENABLE_SINGLETON_FILTER,'min_print_run':PRIMARY_MIN_PRINT_RUN,'overlay_alpha':OVERLAY_ALPHA,'overlay_review_radius':OVERLAY_REVIEW_RADIUS,
}
(OUT/'primary_policy_config.json').write_text(json.dumps(policy_config,indent=2))
print(json.dumps(primary_summary,indent=2))


## 8. Create V8.2a pages and red/green/amber decision overlays


In [ ]:
GREEN=(0,255,0); RED=(0,0,255); AMBER=(0,191,255)

def blend_classes(gray,annotation_mask,uncertain_mask,print_mask,alpha=OVERLAY_ALPHA):
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR); colour=base.copy(); active=np.zeros(gray.shape,bool)
    for mask,bgr in [(annotation_mask,RED),(uncertain_mask,AMBER),(print_mask,GREEN)]:
        colour[mask]=bgr; active|=mask
    out=base.copy(); out[active]=cv2.addWeighted(base,1-alpha,colour,alpha,0)[active]
    return out

def add_legend(image,title):
    h,w=image.shape[:2]; banner=max(42,int(h*0.025)); canvas=np.full((h+banner,w,3),255,np.uint8); canvas[banner:]=image
    scale=max(.45,min(1.0,w/1400)); y=int(banner*.68); cv2.putText(canvas,title,(12,y),cv2.FONT_HERSHEY_SIMPLEX,scale,(40,40,40),1,cv2.LINE_AA)
    x=max(12,w-520); items=[('PRINT',GREEN),('ANNOTATION',RED),('UNCERTAIN',AMBER)]
    for label,bgr in items:
        cv2.rectangle(canvas,(x,y-15),(x+18,y+3),bgr,-1); cv2.putText(canvas,label,(x+24,y+2),cv2.FONT_HERSHEY_SIMPLEX,scale*.72,(40,40,40),1,cv2.LINE_AA); x+=160
    return canvas

integrity=[]
for page,g in primary_rows.groupby('page'):
    src=SOURCE/'whole_pages'/page
    required=['observed.png','v7_8.png','v8_2.png','difference.png','decision_map.png']
    missing=[x for x in required if not (src/x).exists()]
    if missing: raise RuntimeError(f'{page}: source V8.2 whole-page files missing: {missing}')
    observed=cv2.imread(str(src/'observed.png'),0); v78=cv2.imread(str(src/'v7_8.png'),0); v82_original=cv2.imread(str(src/'v8_2.png'),0)
    if observed is None or v78 is None or observed.shape!=v78.shape: raise RuntimeError(f'{page}: unreadable/mismatched source page images')
    out=v78.copy(); decisions=g.decision.to_numpy(); ys=g.y.to_numpy(int); xs=g.x.to_numpy(int)
    pm=np.zeros(observed.shape,bool); am=np.zeros(observed.shape,bool); um=np.zeros(observed.shape,bool)
    pm[ys[decisions=='PRINT'],xs[decisions=='PRINT']]=True; am[ys[decisions=='ANNOTATION'],xs[decisions=='ANNOTATION']]=True; um[ys[decisions=='UNCERTAIN'],xs[decisions=='UNCERTAIN']]=True
    out[pm]=observed[pm]
    authoritative_mask=mask_maps[page]
    outside=int(np.sum(out[~authoritative_mask]!=observed[~authoritative_mask])); nonprint=int(np.sum(out[~pm]!=v78[~pm])); wrong=int(np.sum((out!=v78)&(out!=observed)))
    if outside or nonprint or wrong: raise RuntimeError(f'{page}: integrity failure outside={outside}, nonprint={nonprint}, wrong={wrong}')
    exact=blend_classes(observed,am,um,pm)
    if OVERLAY_REVIEW_RADIUS>0:
        k=np.ones((2*OVERLAY_REVIEW_RADIUS+1,2*OVERLAY_REVIEW_RADIUS+1),np.uint8)
        amp=cv2.dilate(am.astype(np.uint8),k)>0; ump=cv2.dilate(um.astype(np.uint8),k)>0; pmp=cv2.dilate(pm.astype(np.uint8),k)>0
    else: amp,ump,pmp=am,um,pm
    review=add_legend(blend_classes(observed,amp,ump,pmp),f'{page} — V8.2a decision overlay')
    print_only=add_legend(blend_classes(observed,np.zeros_like(pm),np.zeros_like(pm),pmp),f'{page} — V8.2a PRINT overlay')
    dm=np.zeros(observed.shape,np.uint8); dm[am]=85; dm[um]=170; dm[pm]=255; diff=np.zeros_like(observed); diff[out!=v78]=255
    d=WHOLE/page; d.mkdir(parents=True,exist_ok=False)
    cv2.imwrite(str(d/'observed.png'),observed); cv2.imwrite(str(d/'v7_8.png'),v78); cv2.imwrite(str(d/'v8_2_original.png'),v82_original); cv2.imwrite(str(d/'v8_2a.png'),out); cv2.imwrite(str(d/'difference_from_v78.png'),diff); cv2.imwrite(str(d/'decision_map.png'),dm); cv2.imwrite(str(d/'decision_overlay_exact.png'),exact); cv2.imwrite(str(d/'decision_overlay_review.png'),review)
    if GENERATE_PRINT_ONLY_OVERLAY: cv2.imwrite(str(d/'decision_overlay_print_only.png'),print_only)
    page_metric=primary_pages[primary_pages.page==page].iloc[0].to_dict(); page_metric.update({'outside_mask_drift':outside,'non_print_v78_regression':nonprint,'changed_not_equal_observed':wrong})
    (d/'page_metrics.json').write_text(json.dumps(page_metric,indent=2,default=str))
    integrity.append({'page':page,'outside_mask_drift':outside,'non_print_v78_regression':nonprint,'changed_not_equal_observed':wrong,'observed_sha256':sha256_file(d/'observed.png'),'v78_sha256':sha256_file(d/'v7_8.png'),'v82_original_sha256':sha256_file(d/'v8_2_original.png'),'v82a_sha256':sha256_file(d/'v8_2a.png'),'overlay_exact_sha256':sha256_file(d/'decision_overlay_exact.png'),'overlay_review_sha256':sha256_file(d/'decision_overlay_review.png')})

pd.DataFrame(integrity).to_csv(AUDIT/'primary_page_integrity_and_hashes.csv',index=False)
print('V8.2a whole pages and overlays:',len(integrity))


## 9. Visual-audit contact sheets for the primary exploratory policy


In [ ]:
def make_sheet(rows,name,max_pages=12):
    rows=rows.head(max_pages)
    if len(rows)==0:return
    fig,axes=plt.subplots(len(rows),3,figsize=(15,max(5,6*len(rows))),squeeze=False)
    for r,(_,row) in enumerate(rows.iterrows()):
        page=row.page
        paths=[WHOLE/page/'v7_8.png',WHOLE/page/'v8_2a.png',WHOLE/page/'decision_overlay_review.png']
        titles=[f'{page} V7.8',f'{page} V8.2a',f'{page} decisions']
        for c,(path,title) in enumerate(zip(paths,titles)):
            im=cv2.imread(str(path),cv2.IMREAD_UNCHANGED)
            if im is not None and im.ndim==3: im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
            axes[r,c].imshow(im,cmap='gray' if im is not None and im.ndim==2 else None); axes[r,c].set_title(title); axes[r,c].axis('off')
    plt.tight_layout(); plt.savefig(VISUAL/f'{name}.png',dpi=130,bbox_inches='tight'); plt.close(fig)

make_sheet(primary_pages.sort_values(['true_print','utility'],ascending=False),'primary_beneficial_pages')
make_sheet(primary_pages.sort_values(['false_annotation','selected'],ascending=False),'primary_false_heavy_pages')
make_sheet(primary_pages.sort_values('uncertain_rate',ascending=False),'primary_uncertain_pages')
make_sheet(primary_pages.assign(change=lambda x:x.selected).sort_values('change',ascending=False),'primary_most_changed_pages')
print('Visual audit sheets:',VISUAL)


## 10. Final V8.2a identity, source record and concise summary


In [ ]:
environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
(OUT/'software_environment.json').write_text(json.dumps(environment,indent=2))

identity={
    'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.2a-policy-sensitivity','development_only':True,'promotion_authority':False,'retrained':False,'refitted':False,'recalibrated':False,'source_v82_run':str(SOURCE),'source_v82_identity_sha256':sha256_file(SOURCE/'experiment_identity.json'),'source_record':source_record,'primary_policy':json.loads((OUT/'primary_policy_config.json').read_text()),'sweep_print_thresholds':SWEEP_PRINT_THRESHOLDS,'sweep_singleton_filter_options':SWEEP_SINGLETON_FILTER_OPTIONS,'bootstrap_resamples':BOOTSTRAP_RESAMPLES,'copied_source_artifact_inventory_sha256':sha256_file(AUDIT/'copied_source_v82_artifact_inventory.csv'),'primary_predictions_sha256':sha256_file(TABLES/'primary_policy_predictions.parquet'),'threshold_sweep_sha256':sha256_file(TABLES/'threshold_sweep_summary.csv'),'software':environment,
}
(OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))

best_utility=sweep.sort_values(['utility','true_print'],ascending=False).iloc[0].to_dict()
lines=[
    'V8.2a POLICY SENSITIVITY, THRESHOLD SWEEP AND OVERLAY AUDIT','',
    f'Source V8.2 run: {SOURCE}',f'V8.2a run: {OUT}',
    'Retrained: False','Refitted: False','Recalibrated: False','',
    f'Primary policy: {PRIMARY_POLICY_ID}',f'Primary effective PRINT threshold: {effective_print_threshold(PRIMARY_PRINT_THRESHOLD,PRIMARY_PRINT_MARGIN):.6f}',f'Primary selected PRINT pixels: {primary_summary["selected_print_pixels"]}',f'Primary true PRINT: {primary_summary["true_print"]}',f'Primary false annotation: {primary_summary["false_annotation"]}',f'Primary truth-UNCERTAIN PRINT: {primary_summary["truth_uncertain_print"]}',f'Primary utility: {primary_summary["utility"]}',f'Primary catastrophic components: {primary_summary["catastrophic_components"]}',f'Primary benefiting pages: {primary_summary["benefiting_pages"]}',f'Primary benefiting source groups: {primary_summary["benefiting_source_groups"]}','',
    f'Highest observed development utility in sweep: {best_utility.get("policy_id")}',f'Highest observed utility: {best_utility.get("utility")}',f'At true PRINT / false annotation: {best_utility.get("true_print")} / {best_utility.get("false_annotation")}','',
    'Interpretation: retrospective development sensitivity only. No policy from this sweep is frozen or authorised for independent/production use.','All source V8.2 core records, model files and calibrators were copied into this V8.2a run and hash-inventoried.'
]
(OUT/'executive_summary.txt').write_text('\n'.join(lines))
print('\n'.join(lines)); print('\nV8.2a COMPLETE:',OUT)


## Outputs to inspect

Start with:

- `executive_summary.txt`
- `tables/threshold_sweep_summary.csv`
- `visual_audit/threshold_sweep_true_false.png`
- `visual_audit/threshold_sweep_utility.png`
- `tables/probability_quantiles.csv`
- `visual_audit/probability_distribution.png`
- `primary_policy_summary.json`
- `visual_audit/primary_beneficial_pages.png`
- `visual_audit/primary_false_heavy_pages.png`
- `visual_audit/primary_uncertain_pages.png`
- `whole_pages/<page>/decision_overlay_review.png`
- `whole_pages/<page>/decision_overlay_print_only.png`
- `audit/copied_source_v82_artifact_inventory.csv`
- `source_v82_record.json`

For another exploratory primary policy, alter only the block headed **V8.2a TWEAKABLE POLICY CONTROLS** and rerun V8.2a into a new output directory. Do not overwrite the original V8.2 or an earlier V8.2a run.